In [1]:
import pandas as pd
import numpy as np
from scipy import stats

employees = pd.read_csv('employees.csv')
attrition = pd.read_csv('attrition_log.csv')
engagement = pd.read_csv('engagement.csv')

print("Employees:", employees.shape)
print("Attrition:", attrition.shape)
print("Engagement:", engagement.shape)

Employees: (13403, 24)
Attrition: (1400, 10)
Engagement: (55971, 12)


In [5]:
employees["is_leaver"] = employees["status"].str.lower() == "departed"
engagement["survey_date"] = pd.to_datetime(engagement["survey_date"])

print("is_leaver value counts:")
print(employees["is_leaver"].value_counts())

print("\nDepartments:")
print(employees["department"].unique())

print("\nExit types:")
print(attrition["exit_type"].unique())

print('\nEngagement columns')
print(engagement.columns.tolist())

is_leaver value counts:
is_leaver
False    12003
True      1400
Name: count, dtype: int64

Departments:
['Wealth Management' 'Insurance' 'Corporate Operations' 'Technology'
 'Retail Banking' 'Risk & Compliance' 'Executive Leadership']

Exit types:
['voluntary' 'involuntary']

Engagement columns
['employee_id', 'wave_number', 'survey_date', 'response_flag', 'manager_effectiveness', 'psychological_safety', 'recognition', 'career_development', 'senior_leadership_trust', 'purpose_meaning', 'wellbeing', 'confidence_in_role_future']


In [6]:
DEPT_NAME = "Risk & Compliance"
VOLUNTARY_LABEL = "voluntary" 

dimensions = ['manager_effectiveness', 'psychological_safety', 'recognition', 'career_development',
              'senior_leadership_trust', 'purpose_meaning', 'wellbeing', 'confidence_in_role_future']

resp = engagement[engagement["response_flag"]]  # only actual answers for score trends

def first_last_delta(g):
    g = g.sort_values("wave_number")
    if len(g) < 2:
        return pd.Series({d: np.nan for d in dimensions})
    return g[dimensions].iloc[-1] - g[dimensions].iloc[0]

emp_delta = resp.groupby("employee_id").apply(first_last_delta, include_groups=False)

print(f"Employees with a computable trajectory (2+ responded waves): {emp_delta.dropna(how='all').shape[0]}")
emp_delta.head()

Employees with a computable trajectory (2+ responded waves): 11157


,manager_effectiveness,psychological_safety,recognition,career_development,senior_leadership_trust,purpose_meaning,wellbeing,confidence_in_role_future
employee_id,,,,,,,,
E00001,-0.43,-0.05,0.45,-0.47,0.85,0.28,-0.37,-0.20
E00003,0.04,-0.54,-0.48,-0.01,0.06,0.84,0.00,0.00
E00004,0.01,-1.14,-1.86,-0.60,0.23,1.37,0.30,0.03
E00005,0.64,0.41,-0.40,-0.51,0.12,1.05,-0.54,0.94
E00006,-0.20,-0.02,-0.32,-0.43,-0.05,-1.14,-0.82,-1.49


In [7]:
emp_meta = employees[["employee_id", "department"]].merge(
    attrition[["employee_id", "exit_type"]], on="employee_id", how="left"
)

emp_delta = emp_delta.merge(emp_meta, left_index=True, right_on="employee_id", how="left")

emp_delta["is_rc"] = emp_delta["department"] == DEPT_NAME
emp_delta["is_voluntary_leaver"] = emp_delta["exit_type"] == VOLUNTARY_LABEL
emp_delta["is_stayer"] = emp_delta["exit_type"].isna()

print(emp_delta[["employee_id", "department", "exit_type", "is_rc", "is_voluntary_leaver", "is_stayer"]].head())

  employee_id            department  exit_type  is_rc  is_voluntary_leaver  \
0      E00001     Wealth Management        NaN  False                False   
1      E00003     Wealth Management        NaN  False                False   
2      E00004             Insurance        NaN  False                False   
3      E00005     Wealth Management  voluntary  False                 True   
4      E00006  Corporate Operations  voluntary  False                 True   

   is_stayer  
0       True  
1       True  
2       True  
3      False  
4      False  


In [8]:
four_way = emp_delta[emp_delta["is_voluntary_leaver"] | emp_delta["is_stayer"]]

summary = (
    four_way.groupby(["is_rc", "is_voluntary_leaver"])[dimensions]
    .mean()
    .round(3)
)
print(summary)

print("\nSample sizes per group:")
print(four_way.groupby(["is_rc", "is_voluntary_leaver"]).size())

                           manager_effectiveness  psychological_safety  \
is_rc is_voluntary_leaver                                                
False False                               -0.021                -0.031   
      True                                -0.054                -0.095   
True  False                                0.002                 0.015   
      True                                -0.110                -0.033   

                           recognition  career_development  \
is_rc is_voluntary_leaver                                    
False False                     -0.018              -0.012   
      True                      -0.085              -0.027   
True  False                     -0.065               0.011   
      True                       0.037              -0.003   

                           senior_leadership_trust  purpose_meaning  \
is_rc is_voluntary_leaver                                             
False False                             

In [9]:
rc_leaver_deltas = four_way[(four_way["is_rc"]) & (four_way["is_voluntary_leaver"])]
other_leaver_deltas = four_way[(~four_way["is_rc"]) & (four_way["is_voluntary_leaver"])]

print("R&C voluntary leavers vs. rest-of-company voluntary leavers — trajectory delta comparison:\n")
for dim in dimensions:
    a = rc_leaver_deltas[dim].dropna()
    b = other_leaver_deltas[dim].dropna()
    if len(a) < 2 or len(b) < 2:
        print(f"{dim:30s}  not enough data to test (n_rc={len(a)}, n_other={len(b)})")
        continue
    u, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    flag = "  <-- significant at 0.05" if p < 0.05 else ""
    print(f"{dim:30s}  R&C mean delta={a.mean():+.3f} (n={len(a)})  |  "
          f"Other mean delta={b.mean():+.3f} (n={len(b)})  |  p={p:.4f}{flag}")

R&C voluntary leavers vs. rest-of-company voluntary leavers — trajectory delta comparison:

manager_effectiveness           R&C mean delta=-0.110 (n=92)  |  Other mean delta=-0.054 (n=447)  |  p=0.5063
psychological_safety            R&C mean delta=-0.033 (n=92)  |  Other mean delta=-0.095 (n=447)  |  p=0.5210
recognition                     R&C mean delta=+0.037 (n=92)  |  Other mean delta=-0.085 (n=447)  |  p=0.0345  <-- significant at 0.05
career_development              R&C mean delta=-0.003 (n=92)  |  Other mean delta=-0.027 (n=447)  |  p=0.5148
senior_leadership_trust         R&C mean delta=-0.107 (n=92)  |  Other mean delta=-0.099 (n=447)  |  p=0.9110
purpose_meaning                 R&C mean delta=-0.052 (n=92)  |  Other mean delta=-0.068 (n=447)  |  p=0.9806
wellbeing                       R&C mean delta=+0.001 (n=92)  |  Other mean delta=-0.038 (n=447)  |  p=0.6304
confidence_in_role_future       R&C mean delta=-0.090 (n=92)  |  Other mean delta=-0.064 (n=447)  |  p=0.8146


## 6. Interpretation notes

- **Level effect vs. slope effect**: if R&C's *stayers* also show a similarly negative delta
  compared to company-wide stayers, that points to a department-wide morale issue (a level
  effect) — not something specific to people who are about to leave. Worth eyeballing the
  `is_rc=True, is_voluntary_leaver=False` row against `is_rc=False, is_voluntary_leaver=False`
  in the section 4 table for this.
- **Small-n caution**: check the sample sizes printed in section 4 before treating any single
  dimension as a headline finding — R&C voluntary leavers is likely a small group.
- **Next step if a dimension stands out**: cross-reference with the "response rate by proximity
  to exit" trajectory (silence/non-response) from the earlier notebook — if the same dimension
  that shows the steepest score decline is also where non-response climbs before exit, that's a
  strong, twice-confirmed signal rather than a single test result.
